In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer.FisherBufferTrainer import FisherBufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead, print_colored
from src import models
from src.buffer import MultiTaskBuffer

from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_FISHER_BUFFER_CONFIG as CONFIG

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
buffer = MultiTaskBuffer([])
buffer_trainer = FisherBufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    buffer=buffer,
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    buffer_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
        target_acc=0.9,
    )
    buffer_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

    if i < len(test_tasks) - 1:
        buffer_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            context_id=i,
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )

### Domain Incremental Training

In [ ]:
SEED = 0
train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=1)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

sizes = ([600] * 4) + [0]
train_tasks, buffer_tasks = zip(
    *[
        create_holdout_set(dataset, holdout_size=holdout)
        for dataset, holdout in zip(train_tasks, sizes)
    ]
)
print([len(task) for task in buffer_tasks])
print([len(task) for task in train_tasks])

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
buffer = MultiTaskBuffer([])
buffer_trainer = FisherBufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["min_acc_limit"],
    min_acc_increment=CONFIG["min_acc_increment"],
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    buffer=buffer,
    domain_map_fn=domain_map_fn,
    paradigm="DIL",
    seed=SEED,
)

MAX_BUFFER_CALLS = 1
target_acc = 0.7
lower_bounds = []
buffer_calls = []
for i, (train, test, buffer) in enumerate(zip(train_tasks, test_tasks, buffer_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    results = buffer_trainer.test(test_tasks)
    accs = [res[1] for res in results]

    # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
    j = 0
    buffer_call = 0
    prev_acc = None
    while (
        j < MAX_BUFFER_CALLS
        and results[i][1] < target_acc
        and i > 0
        and not buffer_trainer.buffer.is_empty()
    ):
        buffer_call += 1
        print_colored("Using buffer to recompute LID.", color="amber")

        buffer_trainer.recall_dataset, (buffer_X, buffer_y) = (
            buffer_trainer.buffer.consume_merge(buffer_trainer.recall_dataset)
        )
        print("Recall dataset size:", len(buffer_trainer.recall_dataset))
        dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
        buffer_trainer.compute_rashomon_set(
            dataset,
            test_tasks[i + 1] if i + 1 < len(test_tasks) else test_tasks[i],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
        )
        results = buffer_trainer.test(test_tasks)
        accs = [res[1] for res in results]

        print("lower_bounds:", lower_bounds)
        diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
        min_diff_idx = diffs.index(
            min(diffs)
        )  # The assumption is that the task closest to its boundary is the one restricting further improvements
        if results[i][1] > target_acc:
            break
        elif (
            prev_acc is not None
            and results[i][1] - prev_acc < CONFIG["loosening_thresh"]
        ):
            print("Loosening task", min_diff_idx, "bounds.")
            lower_bounds[min_diff_idx] = max(
                lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01
            )
            buffer_trainer.min_acc_limit = lower_bounds
        prev_acc = results[i][1]
        j += 1
    buffer_calls.append(buffer_call)

    print_colored(accs, color="green")

    lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

    buffer_trainer.min_acc_limit = lower_bounds

    if i < len(test_tasks) - 1:
        buffer_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )
        buffer_trainer.add_to_buffer(buffer, task_id=i)

### Class Incremental Learning

In [4]:
SEED = 0
train_tasks, _, test_tasks = get_mnist_tasks(
    seed=SEED, emnist=True, train_val_split_ratio=1
)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [5]:
sizes = [4000, 4000, 4000, 4000, 0]
train_tasks, buffer_tasks = zip(
    *[
        create_holdout_set(dataset, holdout_size=holdout)
        for dataset, holdout in zip(train_tasks, sizes)
    ]
)
print([len(task) for task in buffer_tasks])
print([len(task) for task in train_tasks])

[4000, 4000, 4000, 4000, 0]
[44000, 44000, 44000, 44000, 48000]


In [6]:
buffer = MultiTaskBuffer([])
buffer_trainer = FisherBufferTrainer(
    model,
    checkpoint=CONFIG["checkpoint"],
    n_iters=CONFIG["n_iters"],
    min_acc_limit=CONFIG["initial_target_acc"],
    min_acc_increment=0,
    primal_learning_rate=CONFIG["primal_learning_rate"],
    dual_learning_rate=CONFIG["dual_learning_rate"],
    projection_strategy=CONFIG["projection_strategy"],
    n_certificate_samples=CONFIG["n_certificate_samples"],
    penalty_coefficient=CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
    buffer=buffer,
)

MAX_BUFFER_CALLS = CONFIG["max_buffer_calls"]
target_acc = CONFIG["target_acc"]
lower_bounds = []
buffer_calls = []
for i, (train, test, buffer) in enumerate(zip(train_tasks, test_tasks, buffer_tasks)):
    buffer_trainer.train(
        train,
        test,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    results = buffer_trainer.test(test_tasks)
    accs = [res[1] for res in results]

    # If results are not satisfactory, then use buffer data to recompute rashomon set and continue training
    j = 0
    buffer_call = 0
    prev_acc = None
    while (
        j < MAX_BUFFER_CALLS
        and results[i][1] < target_acc
        and i > 0
        and not buffer_trainer.buffer.is_empty()
    ):
        buffer_trainer.rashomon_kwargs.dual_learning_rate = CONFIG["dual_learning_rate"] / len(lower_bounds) + 1
        buffer_call += 1
        print_colored("Using buffer to recompute LID.", color="amber")

        buffer_trainer.recall_dataset, (buffer_X, buffer_y) = (
            buffer_trainer.buffer.consume_merge(buffer_trainer.recall_dataset)
        )
        print("Recall dataset size:", len(buffer_trainer.recall_dataset))
        dataset = torch.utils.data.TensorDataset(buffer_X, buffer_y)
        buffer_trainer.compute_rashomon_set(
            dataset,
            test_tasks[i + 1] if i < len(test_tasks) else test_tasks[i],
            use_outer_bbox=False,
            batch_size=len(dataset),
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )
        buffer_trainer.train(
            train,
            test,
            batch_size=CONFIG["batch_size"],
            epochs=CONFIG["epochs"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
        )
        results = buffer_trainer.test(test_tasks)
        accs = [res[1] for res in results]

        print("lower_bounds:", lower_bounds)
        diffs = [accs[i] - lower_bounds[i] for i in range(len(lower_bounds))]
        min_diff_idx = diffs.index(
            min(diffs)
        )  # The assumption is that the task closest to its boundary is the one restricting further improvements
        if results[i][1] > target_acc:
            break
        elif (
            prev_acc is not None
            and results[i][1] - prev_acc < CONFIG["loosening_thresh"]
        ):
            print("Loosening task", min_diff_idx, "bounds.")
            lower_bounds[min_diff_idx] = max(
                lower_bounds[min_diff_idx] - CONFIG["loosening_step"], 0.01
            )
            buffer_trainer.min_acc_limit = lower_bounds
        prev_acc = results[i][1]
        j += 1
    buffer_calls.append(buffer_call)

    print_colored(accs, color="green")

    lower_bounds.append(max(accs[i] - CONFIG["min_acc_increment"], 0.01))

    buffer_trainer.min_acc_limit = lower_bounds

    if i < len(train_tasks) - 1:
        buffer_trainer.compute_rashomon_set(
            test,
            test_tasks[i + 1],
            prune_prop=CONFIG["prune_prop"],
            fisher_batch_size=CONFIG["fisher_batch_size"],
            fisher_epochs=CONFIG["fisher_epochs"],
        )
        buffer_trainer.add_to_buffer(buffer, task_id=i, k=CONFIG["buffer_k"])

    else:
        print("Buffer calls:", buffer_calls)


Training Epochs: 100%|███████████████████████| 5/5 [00:45<00:00,  9.01s/it, val_loss=0.0107, val_acc=0.9962, proj=None]


Test Results: [(0.0104, 0.9964), (30.8693, 0.0), (26.2102, 0.0), (36.9512, 0.0), (30.7392, 0.0)]
[0.9964, 0.0, 0.0, 0.0, 0.0]
To keep top 40%, found global Fisher threshold: 0.0341
Initial acc constraint violation: -0.1988 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=1.00


100%|████████████████████████████████████| 200/200 [00:10<00:00, 18.23it/s, size=319.09, obj=0.051, min_soft_acc=0.764]


Final bbox:  Obj=0.05,  Size=319.09,  Min acc hard=0.77,  Min acc soft=0.78
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['9.97', '44.08', '103.13', '136.41', '170.99', '201.30', '232.90', '260.29', '289.86', '319.09']
Checkpoint certificates: ['0.87', '0.98', '0.77', '0.84', '0.79', '0.81', '0.78', '0.81', '0.81', '0.77']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████████| 5/5 [01:07<00:00, 13.45s/it, val_loss=0.5014, val_acc=0.8140, proj=6]


Test Results: [(0.6648, 0.8438), (0.5014, 0.813), (24.4561, 0.0), (33.9982, 0.0), (30.292, 0.0)]
[0.8438, 0.813, 0.0, 0.0, 0.0]
To keep top 40%, found global Fisher threshold: 0.0947
Initial acc constraint violation: -0.1824 (Positive = violated)
Computing Rashomon set within outer box of size: 232.90
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.61
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.79,  Min acc soft=0.80


100%|█████████████████████████████████████| 200/200 [00:11<00:00, 17.90it/s, size=13.52, obj=0.002, min_soft_acc=0.627]


Final bbox:  Obj=0.00,  Size=13.52,  Min acc hard=0.59,  Min acc soft=0.59
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['2.78', '1.87', '5.97', '8.14', '9.20', '10.38', '11.23', '12.34', '12.89', '13.52']
Checkpoint certificates: ['0.55', '0.77', '0.62', '0.61', '0.61', '0.61', '0.61', '0.59', '0.59', '0.59']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|█████████████████████████| 5/5 [01:07<00:00, 13.47s/it, val_loss=23.6697, val_acc=0.0000, proj=9]


KeyboardInterrupt: 